In [1]:
# import sys
# !{sys.executable} -m pip install -U langchain langchain-core langchain-community pydantic duckduckgo-search langchain-experimental

import sys
!{sys.executable} -m pip install -U ddgs


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: D:\Langchain\ModelComponent\venv\Scripts\python.exe -m pip install --upgrade pip


## Built-in Tools - DuckDuckGo Search

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

C:\Users\kuldi\AppData\Local\Temp\ipykernel_13068\2194339046.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


10 minutes ago · Top News · Keralam Advocate General recommends FIR registration against former CM Pinarayi, kin · Mamata moves SC against ECI's freeze on Trinamool Congress ... 6 hours ago · Trump signs Russia sanctions bill into law, allowing up to 100% tariff on India and China · Mamata Banerjee loses symbol, party name, gains Congress support, CJP ... 49 minutes ago · Latest News Today: Follow today's news updates and recent stories from India and across the globe, covering politics, business, technology, and sports. 5 hours ago · India Expresses Deep Concern Over Escalating Security Situation . Tripura Police Arrest Three Bangladeshi Women, Indian National for Alleged Illegal Entry 7 hours ago · 11 PG hostels sealed in Pune for fire-safety violations LIVE! sanctions bill today … seeking 100% tariffs on India. Trader Stabbed In. India pilot under ...


In [4]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


## Custom Tools

In [6]:
from langchain_core.tools import tool

In [7]:
# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [8]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [9]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [10]:
result = multiply.invoke({"a":3, "b":5})

In [11]:
print(result)

15


In [12]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [13]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## Method 2 - Using StructuredTool

In [15]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [16]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

C:\Users\kuldi\AppData\Local\Temp\ipykernel_13068\3279971488.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\kuldi\AppData\Local\Temp\ipykernel_13068\3279971488.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [17]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [18]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [19]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Method 3 - Using BaseTool Class

In [21]:
from langchain.tools import BaseTool
from typing import Type

In [22]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

C:\Users\kuldi\AppData\Local\Temp\ipykernel_13068\908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\kuldi\AppData\Local\Temp\ipykernel_13068\908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [23]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [24]:
multiply_tool = MultiplyTool()

In [25]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Toolkit

In [27]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


In [28]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [29]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)


add => Add two numbers
multiply => Multiply two numbers
